## FABlib API References Examples

- [fablib.show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config)
- [fablib.list_sites](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.list_sites)
- [fablib.list_hosts](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.list_hosts)
- [fablib.new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice)
- [slice.add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node)
- [slice.submit](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.submit)
- [slice.get_nodes](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.get_nodes)
- [slice.list_nodes](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.list_nodesß)
- [slice.show](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.show)
- [node.execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute)
- [slice.delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) 

In [13]:
import datetime
import json
import asyncio

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()

fablib.show_config();

User: apipilikas@gmail.com bastion key is valid!
Configuration is valid


Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
Token File,/home/fabric/.tokens.json
Project ID,49f65ad7-d8a2-4ab9-8ca0-ba777a2e0ea2
Bastion Host,bastion.fabric-testbed.net
Bastion Username,apipilikas_0000444352
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub
Slice Private Key File,/home/fabric/work/fabric_config/slice_key


In [17]:
slice_name = 'S-Drive-on-FABRIC'
image = "default_ubuntu_24"

node_configurations = [
    {
        "type": "control",
        "cores": 2,
        "ram": 8,
        "disk": 100,
        "site": "AMST",
        "host": "amst-w3.fabric-testbed.net",
    },
    {
        "type": "dynamos",
        "cores": 4,
        "ram": 16,
        "disk": 100,
        "site": "AMST",
        "host": "amst-w3.fabric-testbed.net",
    },
    {
        "type": "agent",
        "name": "server",
        "cores": 4,
        "ram": 16,
        "disk": 100,
        "site": "AMST",
        "host": "amst-w3.fabric-testbed.net",
    },
    {
        "type": "agent",
        "name": "clientone",
        "cores": 4,
        "ram": 16,
        "disk": 100,
        "site": "LOSA",
        "host": "losa-w3.fabric-testbed.net",
    },
    {
        "type": "agent",
        "name": "clienttwo",
        "cores": 8,
        "ram": 16,
        "disk": 100,
        "site": "TOKY",
        "host": "toky-w3.fabric-testbed.net",
    },
    {
        "type": "agent",
        "name": "clientthree",
        "cores": 8,
        "ram": 16,
        "disk": 100,
        "site": "AMST",
        "host": "amst-w3.fabric-testbed.net",
    },
    {
        "type": "thirdparty",
        "name": "surf",
        "cores": 8,
        "ram": 16,
        "disk": 100,
        "site": "TOKY",
        "host": "toky-w3.fabric-testbed.net",
    }
]

sites = list(set([configuration["site"] for configuration in node_configurations]))

def create_node(slice, configuration):
    if (configuration["type"] == "control"): 
        configuration["name"] = "control"

    if (configuration["type"] == "dynamos"): 
        configuration["name"] = "dynamos"
    
    return slice.add_node(name=configuration["name"], 
                          site=configuration["site"], 
                          host=configuration["host"], 
                          cores=configuration["cores"], 
                          ram=configuration["ram"], 
                          disk=configuration["disk"], 
                          validate=True, 
                          raise_exception=True, 
                          image=image)
    

Gets a list of available image names.

In [15]:
print(fablib.get_image_names())

{'default_centos8_stream': {'description': 'CentOS 8 Stream (non-stream)', 'default_user': 'centos'}, 'default_centos9_stream': {'description': 'CentOS 9 Stream (default install)', 'default_user': 'cloud-user'}, 'default_centos10_stream': {'description': 'CentOS 10 Stream (default install)', 'default_user': 'cloud-user'}, 'default_debian_11': {'description': 'Debian 11 Bullseye', 'default_user': 'debian'}, 'default_debian_12': {'description': 'Debian 12 Bookworm', 'default_user': 'debian'}, 'default_fedora_39': {'description': 'Fedora 39', 'default_user': 'fedora'}, 'default_fedora_40': {'description': 'Fedora 40', 'default_user': 'fedora'}, 'default_freebsd_13_zfs': {'description': 'FreeBSD 13 with ZFS', 'default_user': 'freebsd'}, 'default_freebsd_14_zfs': {'description': 'FreeBSD 14 with ZFS', 'default_user': 'freebsd'}, 'default_kali': {'description': 'Kali Linux (for penetration testing)', 'default_user': 'kali'}, 'default_openbsd_7': {'description': 'OpenBSD 7', 'default_user': '

In [18]:
# Create a slice
slice = fablib.new_slice(name=slice_name)

# Add Nodes with the specific variables
# Also validate the node can be created and raise an exception in case of failure
print('Adding nodes...')
nodes = [create_node(slice, configuration) for configuration in node_configurations]
nodes_per_site = [
    (site, [node for node in nodes if node.get_site() == site])
    for site in sites
]

print('Adding network interfaces...')
interfaces_per_site = [
    (site, [node.add_component(model='NIC_Basic', name='NIC').get_interfaces()[0] for node in nodes])
    for (site, nodes) in nodes_per_site
]

print('Adding network...')
networks = [
    slice.add_l3network(name=f'Network-{site}', interfaces=interfaces, type="IPv4")
    for (site, interfaces) in interfaces_per_site
]

print(networks, [n.get_gateway() for n in networks], [n.get_subnet() for n in networks])

# Calculate the lease end time for 2 weeks from now with timezone information
lease_end_time = datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(weeks=2)

# Submit the slice, using an end date 2 weeks from now (the current maximum lease time) 
# to make sure that the slice can be used for a longer period of time. Progress shows an indicator of the current progression.
# Wait until the state is finished and use an interval (it may take some time before the slice and nodes are created)
print('Creating slice...')
slice.submit(wait=True, wait_timeout=3600, wait_interval=20, progress=True, wait_jupyter='text', lease_end_time=lease_end_time);


Retry: 11, Time: 316 sec


ID,4806dd3e-31d0-4a78-a460-b6ae77b1249a
Name,S-Drive-on-FABRIC
Lease Expiration (UTC),2026-05-02 11:09:03 +0000
Lease Start (UTC),2026-04-18 11:09:03 +0000
Project ID,49f65ad7-d8a2-4ab9-8ca0-ba777a2e0ea2
State,StableOK
Email,apipilikas@gmail.com
UserId,978ed605-840c-4c61-8296-68e45dbe1c32


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
9e6228de-abb6-41d3-b9c5-a6be58d353b4,clientone,4,16,100,default_ubuntu_24,qcow2,losa-w3.fabric-testbed.net,LOSA,ubuntu,2001:400:a100:3070:f816:3eff:fe07:b439,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3070:f816:3eff:fe07:b439,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
404e9c5b-f677-42bb-a2f6-50515ce3a13c,clientthree,8,16,100,default_ubuntu_24,qcow2,amst-w3.fabric-testbed.net,AMST,ubuntu,2001:610:2d0:fabc:f816:3eff:fe3c:fea2,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe3c:fea2,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
a980fce6-bd85-423a-ba81-03ef554099f0,clienttwo,8,16,100,default_ubuntu_24,qcow2,toky-w3.fabric-testbed.net,TOKY,ubuntu,133.69.160.253,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@133.69.160.253,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
586a72f5-2974-468d-8fc0-2bfa3a484266,control,2,8,100,default_ubuntu_24,qcow2,amst-w3.fabric-testbed.net,AMST,ubuntu,2001:610:2d0:fabc:f816:3eff:fe5e:4fe2,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe5e:4fe2,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
7361228a-afc0-42f8-9225-83ee5181bcb7,dynamos,4,16,100,default_ubuntu_24,qcow2,amst-w3.fabric-testbed.net,AMST,ubuntu,2001:610:2d0:fabc:f816:3eff:fe80:2609,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe80:2609,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
37c48c67-d8d1-46e7-a495-cd9a20164f46,server,4,16,100,default_ubuntu_24,qcow2,amst-w3.fabric-testbed.net,AMST,ubuntu,2001:610:2d0:fabc:f816:3eff:fe5d:66b2,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe5d:66b2,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
d81c44d0-ee00-4778-8624-327a89be4ea1,surf,8,16,100,default_ubuntu_24,qcow2,toky-w3.fabric-testbed.net,TOKY,ubuntu,133.69.160.220,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@133.69.160.220,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key


ID,Name,Layer,Type,Site,Subnet,Gateway,State,Error
20b2e1b4-6ca8-4f3c-8aa4-ed78c260c027,Network-AMST,L3,FABNetv4,AMST,10.145.3.0/24,10.145.3.1,Active,
248d71ad-32f6-4425-aff2-9fa9745f1c6c,Network-LOSA,L3,FABNetv4,LOSA,10.137.1.0/24,10.137.1.1,Active,
5cbc1621-dd32-45a6-9b2c-d3457d708473,Network-TOKY,L3,FABNetv4,TOKY,10.146.2.0/24,10.146.2.1,Active,


Name,Short Name,Node,Network,Bandwidth,Mode,VLAN,MAC,Physical Device,Device,IP Address,Numa Node,Switch Port
control-NIC-p1,p1,control,Network-AMST,100,config,,0A:40:F0:AF:E6:92,enp7s0,enp7s0,fe80::840:f0ff:feaf:e692,4,HundredGigE0/0/0/9
dynamos-NIC-p1,p1,dynamos,Network-AMST,100,config,,0A:72:73:1C:13:2A,enp7s0,enp7s0,fe80::872:73ff:fe1c:132a,4,HundredGigE0/0/0/9
server-NIC-p1,p1,server,Network-AMST,100,config,,0A:B1:A5:3F:0F:02,enp7s0,enp7s0,fe80::8b1:a5ff:fe3f:f02,4,HundredGigE0/0/0/9
clientone-NIC-p1,p1,clientone,Network-LOSA,100,config,,06:BA:43:20:27:43,enp6s0,enp6s0,fe80::4ba:43ff:fe20:2743,4,HundredGigE0/0/0/9
clienttwo-NIC-p1,p1,clienttwo,Network-TOKY,100,config,,06:10:C8:B2:1E:58,enp7s0,enp7s0,fe80::410:c8ff:feb2:1e58,4,HundredGigE0/0/0/9
clientthree-NIC-p1,p1,clientthree,Network-AMST,100,config,,0E:4F:18:21:9F:35,enp7s0,enp7s0,fe80::c4f:18ff:fe21:9f35,4,HundredGigE0/0/0/9
surf-NIC-p1,p1,surf,Network-TOKY,100,config,,06:CB:A9:6F:7D:11,enp7s0,enp7s0,fe80::4cb:a9ff:fe6f:7d11,4,HundredGigE0/0/0/9



Time to print interfaces 331 seconds


In [19]:
slice = fablib.get_slice(name=slice_name);
nodes = slice.get_nodes();
nodes_and_network_per_site = [
    (site, [node for node in nodes if node.get_site() == site], slice.get_network(name=f"Network-{site}"))
    for site in sites
]

nodes_network_ips_per_site = [
    (site, nodes, network, network.get_available_ips(len(nodes)))
    for (site, nodes, network) in nodes_and_network_per_site
]

In [20]:
slice.list_nodes();

ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
9e6228de-abb6-41d3-b9c5-a6be58d353b4,clientone,4,16,100,default_ubuntu_24,qcow2,losa-w3.fabric-testbed.net,LOSA,ubuntu,2001:400:a100:3070:f816:3eff:fe07:b439,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3070:f816:3eff:fe07:b439,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
404e9c5b-f677-42bb-a2f6-50515ce3a13c,clientthree,8,16,100,default_ubuntu_24,qcow2,amst-w3.fabric-testbed.net,AMST,ubuntu,2001:610:2d0:fabc:f816:3eff:fe3c:fea2,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe3c:fea2,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
a980fce6-bd85-423a-ba81-03ef554099f0,clienttwo,8,16,100,default_ubuntu_24,qcow2,toky-w3.fabric-testbed.net,TOKY,ubuntu,133.69.160.253,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@133.69.160.253,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
586a72f5-2974-468d-8fc0-2bfa3a484266,control,2,8,100,default_ubuntu_24,qcow2,amst-w3.fabric-testbed.net,AMST,ubuntu,2001:610:2d0:fabc:f816:3eff:fe5e:4fe2,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe5e:4fe2,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
7361228a-afc0-42f8-9225-83ee5181bcb7,dynamos,4,16,100,default_ubuntu_24,qcow2,amst-w3.fabric-testbed.net,AMST,ubuntu,2001:610:2d0:fabc:f816:3eff:fe80:2609,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe80:2609,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
37c48c67-d8d1-46e7-a495-cd9a20164f46,server,4,16,100,default_ubuntu_24,qcow2,amst-w3.fabric-testbed.net,AMST,ubuntu,2001:610:2d0:fabc:f816:3eff:fe5d:66b2,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe5d:66b2,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
d81c44d0-ee00-4778-8624-327a89be4ea1,surf,8,16,100,default_ubuntu_24,qcow2,toky-w3.fabric-testbed.net,TOKY,ubuntu,133.69.160.220,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@133.69.160.220,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key


In [21]:
for node in slice.get_nodes():
    stdout, stderr = node.execute('echo Hello, FABRIC from node `hostname -s`')

Hello, FABRIC from node control
Hello, FABRIC from node dynamos
Hello, FABRIC from node server
Hello, FABRIC from node clientone
Hello, FABRIC from node clienttwo
Hello, FABRIC from node clientthree
Hello, FABRIC from node surf


In [22]:
def assign_ip(site, network, available_ips, node):
    interface = node.get_interface(network_name=f"Network-{site}")
    address = available_ips.pop(0)
    network_gateway = network.get_gateway()
    network_subnet = network.get_subnet()

    network.allocate_ip(address)
    interface.ip_addr_add(addr=address, subnet=network_subnet)
    node.ip_route_add(subnet=network_subnet, gateway=network_gateway)

    # For the multisite IPv4 connection
    for network in networks:
        node.ip_route_add(subnet=network.get_subnet(), gateway=network_gateway)

    return address

ips = [assign_ip(site, network, ips, node) for (site, nodes, network, ips) in nodes_network_ips_per_site for node in nodes];

print(ips);


[IPv4Address('10.137.1.2'), IPv4Address('10.145.3.2'), IPv4Address('10.145.3.3'), IPv4Address('10.145.3.4'), IPv4Address('10.145.3.5'), IPv4Address('10.146.2.2'), IPv4Address('10.146.2.3')]


In [23]:
for node in nodes:
    ssh_command = node.get_ssh_command().replace(
        "-i /home/fabric/work/fabric_config/slice_key", "-i ~/.ssh/keys/FABRIC-slice_key"
    ).replace(
        "-F /home/fabric/work/fabric_config/ssh_config ", ""
    )
    
    print(ssh_command);

ssh -i ~/.ssh/keys/FABRIC-slice_key ubuntu@2001:610:2d0:fabc:f816:3eff:fe5e:4fe2
ssh -i ~/.ssh/keys/FABRIC-slice_key ubuntu@2001:610:2d0:fabc:f816:3eff:fe80:2609
ssh -i ~/.ssh/keys/FABRIC-slice_key ubuntu@2001:610:2d0:fabc:f816:3eff:fe5d:66b2
ssh -i ~/.ssh/keys/FABRIC-slice_key ubuntu@2001:400:a100:3070:f816:3eff:fe07:b439
ssh -i ~/.ssh/keys/FABRIC-slice_key ubuntu@133.69.160.253
ssh -i ~/.ssh/keys/FABRIC-slice_key ubuntu@2001:610:2d0:fabc:f816:3eff:fe3c:fea2
ssh -i ~/.ssh/keys/FABRIC-slice_key ubuntu@133.69.160.220


In [24]:
for node in nodes:
    print(node.get_name(), node.get_interface(network_name=f"Network-{node.get_site()}"))

control ---------------  ------------------
Name             control-NIC-p1
Network          Network-AMST
Bandwidth        100
Mode             config
VLAN
MAC              0A:40:F0:AF:E6:92
Physical Device  enp7s0
Device           enp7s0
Address          10.145.3.2
Numa Node        4
Switch Port      HundredGigE0/0/0/9
---------------  ------------------
dynamos ---------------  ------------------
Name             dynamos-NIC-p1
Network          Network-AMST
Bandwidth        100
Mode             config
VLAN
MAC              0A:72:73:1C:13:2A
Physical Device  enp7s0
Device           enp7s0
Address          10.145.3.3
Numa Node        4
Switch Port      HundredGigE0/0/0/9
---------------  ------------------
server ---------------  ------------------
Name             server-NIC-p1
Network          Network-AMST
Bandwidth        100
Mode             config
VLAN
MAC              0A:B1:A5:3F:0F:02
Physical Device  enp7s0
Device           enp7s0
Address          10.145.3.4
Numa Node        4
